# Assignment 2 - RGB to grayscale

> Write a program to convert an RGB (colour) image to a grayscale image.
> - **Case 1:** By taking the mean average of the three planes.
> - **Case 2:** Input the weightage of the three planes R, G, B from the user. Each weight is
>   between 0 and 1 and the three sum to 1. (Example: 0.7, 0.2, 0.1.)

### What you actually learn here
- indexing the channel axis, and how to collapse it
- `axis=` in numpy reductions - the thing everybody gets wrong once
- why a naive `(r + g + b) / 3` on `uint8` gives garbage
- why the *real* luma weights are 0.299 / 0.587 / 0.114 and not 1/3 each
- validating user input at the boundary instead of trusting it

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_gray_mean, check_gray_weighted

rgb = cv2.cvtColor(cv2.imread("images/fruits.jpg"), cv2.COLOR_BGR2RGB)
print(f"rgb: shape={rgb.shape}  dtype={rgb.dtype}  min={rgb.min()}  max={rgb.max()}")
R, G, B = rgb[..., 0], rgb[..., 1], rgb[..., 2]
_imgs, _titles = [rgb, R, G, B], ["RGB", "R plane", "G plane", "B plane"]
fig, axes = plt.subplots(1, 4, figsize=(3.2 * 4, 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Look at those three planes. The red fruit is *bright* in the R plane and *dark* in the B plane.
That is the whole point of this assignment: grayscale is a choice about how much each colour
contributes, and different choices give visibly different images.

## Collapsing the channel axis

`rgb` has shape `(H, W, 3)`. You want `(H, W)`. Three equivalent framings:

```python
rgb[..., 0]                  # take one channel      -> (H, W)
rgb.mean(axis=2)             # average over channels -> (H, W)  float64
rgb.mean(axis=(0, 1))        # average over PIXELS   -> (3,)     <- not what you want
```

`axis` names the axis that **disappears**. Get this wrong and the shape tells you immediately.

## The overflow trap, concretely

`R + G + B` where all three are `uint8` overflows at 255 and wraps. Watch:

In [ ]:
px = np.array([200, 100, 90], dtype=np.uint8)     # a real-ish bright pixel
print("uint8 sum   :", px[0] + px[1] + px[2], " <- should be 390")
print("float sum   :", px.astype(np.float64).sum())
print("correct mean:", px.astype(np.float64).mean())
print()
print("shape check :", rgb.mean(axis=2).shape, "vs", rgb.mean(axis=(0,1)).shape)

---
## Task 1 - plain mean of the three planes

Return `uint8`, shape `(H, W)`. **Round** to nearest, do not truncate.

In [ ]:
def my_cvtColor_mean(rgb):
    """Grayscale by averaging the three colour planes equally.

    Parameters
    ----------
    rgb : (H, W, 3) uint8 array

    Returns
    -------
    (H, W) uint8 array. Each output pixel is the rounded mean of its three channels.
    """
    # TODO: your code here
    raise NotImplementedError

In [ ]:
check_gray_mean(my_cvtColor_mean)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# There is NO cv2 call for an equal-weight mean -- cv2.cvtColor always uses luma weights.
# The general "weighted sum of channels" call is cv2.transform:
#   cv2.transform(src, m)  where m is a (1,3) or (3,3) matrix
import cv2, numpy as np
cv_mean = cv2.transform(rgb, np.array([[1/3, 1/3, 1/3]]))
print("cv2.transform output shape:", cv_mean.shape, "dtype:", cv_mean.dtype)
print("max difference vs yours:",
      int(np.abs(cv_mean.astype(int) - my_cvtColor_mean(rgb).astype(int)).max()), "(rounding only)")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

One line. Average over the channel axis, `.round()`, then cast to `np.uint8`.
`.mean()` already returns float64, so there is no overflow to worry about *if* you let it do the summing.

</details>

In [ ]:
g = my_cvtColor_mean(rgb)
_imgs, _titles = [rgb, g], ["RGB", "mean grayscale"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
plt.hist(g.ravel(), bins=256, range=(0, 256), color="steelblue")
plt.title("mean grayscale my_calcHist"); plt.xlabel("intensity"); plt.ylabel("pixel count")
plt.xlim(0, 255); plt.tight_layout(); plt.show()

---
## Task 2 - user-supplied weights

The user hands you three numbers. Users are wrong sometimes, so this function has a second job:
**reject invalid weights**. The grader requires that you `raise` (any exception type) when

- any weight is outside `[0, 1]`, or
- the three do not sum to 1

Use a tolerance for the sum - `0.299 + 0.587 + 0.114` is not exactly `1.0` in binary floating
point. Something like `abs(total - 1) > 1e-6`.

In [ ]:
def my_cvtColor_weighted(rgb, wr, wg, wb):
    """Grayscale as a weighted sum of the three planes: out = wr*R + wg*G + wb*B.

    Parameters
    ----------
    rgb        : (H, W, 3) uint8 array
    wr, wg, wb : floats in [0, 1] summing to 1

    Returns
    -------
    (H, W) uint8 array, rounded.

    Raises
    ------
    ValueError if any weight is outside [0, 1] or the three do not sum to 1
               (allow a small floating-point tolerance on the sum).
    """
    # TODO: validate the weights first, then compute
    raise NotImplementedError

In [ ]:
check_gray_weighted(my_cvtColor_weighted)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.cvtColor(src, code)  -- the single most used function in OpenCV.
# COLOR_BGR2GRAY hardcodes the luma weights 0.299 / 0.587 / 0.114.
import cv2, numpy as np
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
cv_gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
mine = my_cvtColor_weighted(rgb, 0.299, 0.587, 0.114)
print("max difference vs yours:", int(np.abs(cv_gray.astype(int) - mine.astype(int)).max()))
# For ARBITRARY weights, cv2.transform is the right call:
cv_w = cv2.transform(rgb, np.array([[0.7, 0.2, 0.1]]))
print("cv2.transform vs yours (0.7/0.2/0.1):",
      int(np.abs(cv_w.astype(int) - my_cvtColor_weighted(rgb, 0.7, 0.2, 0.1).astype(int)).max()))
print("\nCodes to memorise: COLOR_BGR2GRAY, COLOR_BGR2RGB, COLOR_BGR2HSV,")
print("                   COLOR_BGR2YCrCb, COLOR_BGR2LAB, and every 2-way inverse.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Validation: two `if` statements and a `raise ValueError(...)` each.

Computation: you can multiply `rgb` by `np.array([wr, wg, wb])` and numpy will broadcast the 3-vector across the last axis, then sum over `axis=2`. Or do `wr*R + wg*G + wb*B` on float versions of the planes. Both are fine.

</details>

In [ ]:
# The weights broadcasting rule, made visible.
w = np.array([0.299, 0.587, 0.114])
print("rgb shape:", rgb.shape, " w shape:", w.shape)
print("product  :", (rgb * w).shape, "  <- w stretched across H and W automatically")
print("summed   :", (rgb * w).sum(axis=2).shape)

### Why 0.299 / 0.587 / 0.114?

Human eyes are most sensitive to green and least to blue. Equal weights therefore make
foliage look too dark and skies too light. The ITU-R BT.601 luma weights above match
perceived brightness, and they are exactly what `cv2.cvtColor(..., COLOR_BGR2GRAY)` uses.
Compare all of it:

In [ ]:
variants = {
    "mean (1/3 each)"      : my_cvtColor_mean(rgb),
    "luma .299/.587/.114"  : my_cvtColor_weighted(rgb, 0.299, 0.587, 0.114),
    "cv2.cvtColor"         : cv2.cvtColor(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY),
    "red-heavy .7/.2/.1"   : my_cvtColor_weighted(rgb, 0.7, 0.2, 0.1),
    "green only"           : my_cvtColor_weighted(rgb, 0.0, 1.0, 0.0),
}
_imgs, _titles = list(variants.values()), list(variants.keys())
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.2 * _cols, 3.2 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

luma = variants["luma .299/.587/.114"]
cvg  = variants["cv2.cvtColor"]
print("max |yours - cv2| =", int(np.abs(luma.astype(int) - cvg.astype(int)).max()),
      "(1 or 2 is just rounding)")
print("mean brightness: equal-weights", variants["mean (1/3 each)"].mean().round(1),
      " vs luma", luma.mean().round(1))

---
## Think about it

1. Pick the image region where equal-weight and luma grayscale differ the most. Compute the
   absolute difference image and `show` it. Which colour dominates that region? Does the sign
   of the difference match what the weights predict?
2. The assignment insists the weights sum to 1. What visibly happens if they sum to 1.5?
   Bypass your own validation to find out, then explain it in terms of dynamic range.
3. Weights `(1, 0, 0)` reduce to "just take the R plane". Give a real situation where that
   is the *right* grayscale conversion.
4. Grayscale throws away information irreversibly. How many distinct RGB colours map to
   grayscale value 100 under equal weights? Estimate it, then count it exactly with numpy.

In [ ]:
# Scratch space.

---
## Stretch

1. **Contrast as a metric.** Compute the standard deviation of each grayscale variant above.
   Which weighting preserves the most contrast on `fruits.jpg`? On `baboon.jpg`? Same answer?
2. **Colour spaces.** Convert to HSV (`cv2.COLOR_RGB2HSV`) and display the V channel next to
   your luma grayscale. V is `max(R,G,B)`, not a weighted sum. Where do they disagree, and why?
3. **A dedicated channel mixer.** Write `mix(rgb, w)` taking a length-3 array, then find weights
   that make the red fruit and the green leaves in `fruits.jpg` come out at nearly the same
   gray level - the classic way to demonstrate why colourblind-unfriendly figures happen.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - colour conversion

```python
dst = cv2.cvtColor(src, code)
```
The workhorse. `code` is a `cv2.COLOR_*` constant naming source and destination:

| code | note |
|---|---|
| `COLOR_BGR2GRAY` | luma weights 0.299 / 0.587 / 0.114, hardcoded |
| `COLOR_BGR2RGB` | the channel swap you need before every `plt.imshow` |
| `COLOR_BGR2HSV` | H is **0..179** for uint8, not 0..359 |
| `COLOR_BGR2YCrCb` | order is Y, **Cr**, Cb |
| `COLOR_BGR2LAB`, `COLOR_BGR2XYZ`, `COLOR_BGR2LUV` | perceptual / device-independent |
| `COLOR_GRAY2BGR` | replicates one channel into three |

Every code has an inverse (`COLOR_HSV2BGR`, ...). OpenCV assumes **BGR**, so a code
starting `RGB2` is only correct if you really do hold RGB.

```python
dst = cv2.transform(src, m)          # arbitrary channel mix; m is (1,3) or (3,3)
b, g, r = cv2.split(src)             # slow -- prefer src[..., 0] when you can
dst = cv2.merge([b, g, r])
dst = cv2.mixChannels([src], [dst], fromTo)
```